# MLP Baseline — Malware Detection with PyTorch

## 1. Data Integration

### Objective
This section prepares the preprocessed BODMAS dataset for use with a PyTorch Multi-Layer Perceptron (MLP).

### Steps
1. Load the preprocessed training and testing splits.
2. Verify the dimensions and data types.
3. Check the feature and label consistency.
4. Convert the data into PyTorch-compatible tensors.
5. Prepare the data for the MLP training stage.

**Dataset:** BODMAS  
**Framework:** PyTorch  
**Task:** Binary malware classification

### Environment and Data Requirements

Use a Jupyter Python kernel with NumPy and PyTorch installed. Set up the kernel separately on Windows, Linux, or macOS with package versions compatible with your Python version and platform.

Start the baseline kernel from the repository root or from `notebooks/02_baseline_models/`. The baseline reads four files from `data/processed/bodmas/`. The `data/` directory is excluded from Git, so a fresh clone needs the files placed there before running the baseline.

- `X_train.npy` and `X_test.npy` contain `float32` feature matrices.
- `y_train.npy` and `y_test.npy` contain `int32` label vectors.

To regenerate the processed files, obtain the official `bodmas.npz` feature archive and place it at `data/raw/bodmas/bodmas.npz`. Use a kernel with NumPy, pandas, scikit-learn, and joblib. Open [the preprocessing notebook](../01_data/02_data_preprocessing.ipynb) and run it with `notebooks/01_data/` as the kernel's working directory. It saves the four arrays to `data/processed/bodmas/`. [Official BODMAS feature archive](https://whyisyoung.github.io/BODMAS/#download)

In [30]:
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset, TensorDataset

print("PyTorch:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())
print("NumPy:", np.__version__)

PyTorch: 2.15.0
MPS available: True
NumPy: 2.4.6


### 1.1 Loading the Preprocessed Dataset

The BODMAS dataset has already been preprocessed and divided into training and testing sets.

In this step, we load the four NumPy arrays (`X_train`, `X_test`, `y_train`, and `y_test`) from the `data/processed/bodmas/` directory.

We use memory mapping to avoid loading the entire dataset into RAM unnecessarily.

In [31]:
project_root = Path.cwd()
notebook_dir = Path("notebooks/02_baseline_models")
if project_root == project_root.parent.parent / notebook_dir:
    project_root = project_root.parent.parent

data_dir = project_root / "data" / "processed" / "bodmas"
if not data_dir.is_dir():
    raise FileNotFoundError(
        f"Processed BODMAS directory not found at {data_dir}. "
        "Start the kernel from the repository root or notebooks/02_baseline_models/ "
        "and place the four processed files in data/processed/bodmas/."
    )

X_train = np.load(data_dir / "X_train.npy", mmap_mode="r")
X_test = np.load(data_dir / "X_test.npy", mmap_mode="r")
y_train = np.load(data_dir / "y_train.npy", mmap_mode="r")
y_test = np.load(data_dir / "y_test.npy", mmap_mode="r")

print("All four dataset splits loaded successfully!")

All four dataset splits loaded successfully!


### 1.2 Dataset Integrity and Shape

Before creating datasets or loaders, validate both splits. Each split must have a nonempty `float32` feature matrix with 2,381 columns and a matching one-dimensional `int32` label vector containing both binary classes. The validation reads features in chunks to check for NaN and infinity, then reports the shapes and dtypes.

In [32]:
def check_finite_in_chunks(data, chunk_size=1024):
    for start in range(0, len(data), chunk_size):
        chunk = data[start:start + chunk_size]

        if not np.isfinite(chunk).all():
            return False

    return True


def validate_split(X, y, split_name):
    if X.ndim != 2:
        raise ValueError(f"{split_name}: expected 2D features, found shape {X.shape}")
    if y.ndim != 1:
        raise ValueError(f"{split_name}: expected 1D labels, found shape {y.shape}")
    if len(X) == 0 or len(y) == 0:
        raise ValueError(f"{split_name}: features and labels must both be nonempty")
    if X.shape[1] != 2381:
        raise ValueError(f"{split_name}: expected 2381 features, found {X.shape[1]}")
    if len(X) != len(y):
        raise ValueError(f"{split_name}: found {len(X)} feature rows and {len(y)} labels")

    if X.dtype != np.float32:
        raise TypeError(f"{split_name}: expected float32 features, found {X.dtype}")
    if y.dtype != np.int32:
        raise TypeError(f"{split_name}: expected int32 labels, found {y.dtype}")

    labels = np.unique(y)
    if not np.array_equal(labels, [0, 1]):
        raise ValueError(f"{split_name}: expected classes 0 and 1, found {labels.tolist()}")
    if not check_finite_in_chunks(X):
        raise ValueError(f"{split_name}: features contain NaN or infinity")

    print(f"{split_name}: All validation checks passed!")


validate_split(X_train, y_train, "Training")
validate_split(X_test, y_test, "Testing")

Training: All validation checks passed!
Testing: All validation checks passed!


In [33]:
print("X_train:", X_train.shape, X_train.dtype)
print("y_train:", y_train.shape, y_train.dtype)

print("X_test:", X_test.shape, X_test.dtype)
print("y_test:", y_test.shape, y_test.dtype)

print("Number of input features:", X_train.shape[1])

X_train: (107548, 2381) float32
y_train: (107548,) int32
X_test: (26887, 2381) float32
y_test: (26887,) int32
Number of input features: 2381


### 1.3 Label Distribution

We report whether feature and label row counts match and show the class distribution for each split. The class counts help identify imbalance that can affect model evaluation.

In [34]:
print("Training samples match labels:",
      len(X_train) == len(y_train))

print("Testing samples match labels:",
      len(X_test) == len(y_test))

train_labels, train_counts = np.unique(y_train, return_counts=True)
test_labels, test_counts = np.unique(y_test, return_counts=True)

print("\nTraining label distribution:")
print(dict(zip(train_labels.tolist(), train_counts.tolist())))

print("\nTesting label distribution:")
print(dict(zip(test_labels.tolist(), test_counts.tolist())))

Training samples match labels: True
Testing samples match labels: True

Training label distribution:
{0: 61714, 1: 45834}

Testing label distribution:
{0: 15428, 1: 11459}


### 1.4 PyTorch Tensor Preparation

PyTorch tensors are the numerical data structures used by neural networks.

In this step, we verify that the preprocessed features can be converted into PyTorch tensors. We initially use a small sample to avoid unnecessary memory allocation.

The full dataset will later be accessed through a PyTorch Dataset and DataLoader for batch-based training.

In [35]:
X_sample = np.array(X_train[:5], copy=True)
y_sample = np.array(y_train[:5], copy=True)

# Convert to PyTorch tensors
X_tensor = torch.from_numpy(X_sample)
y_tensor = torch.from_numpy(y_sample)

print("Feature tensor shape:", X_tensor.shape)
print("Feature tensor dtype:", X_tensor.dtype)

print("Label tensor shape:", y_tensor.shape)
print("Label tensor dtype:", y_tensor.dtype)

Feature tensor shape: torch.Size([5, 2381])
Feature tensor dtype: torch.float32
Label tensor shape: torch.Size([5])
Label tensor dtype: torch.int32


### 1.5 Understanding PyTorch DataLoader

A PyTorch DataLoader organizes dataset samples into smaller batches.

Batching makes it possible to train the MLP without transferring the entire dataset to the GPU at once.

We first demonstrate batching using five samples before preparing the complete training and testing datasets.

In [36]:
# Prepare labels for binary classification
y_sample_float = y_tensor.float().unsqueeze(1)

# Pair features with their labels
sample_dataset = TensorDataset(X_tensor, y_sample_float)

# Create batches of 2 examples
sample_loader = DataLoader(
    sample_dataset,
    batch_size=2,
    shuffle=False
)

for batch_X, batch_y in sample_loader:
    print("Features:", batch_X.shape)
    print("Labels:", batch_y.shape)
    print("---")

Features: torch.Size([2, 2381])
Labels: torch.Size([2, 1])
---
Features: torch.Size([2, 2381])
Labels: torch.Size([2, 1])
---
Features: torch.Size([1, 2381])
Labels: torch.Size([1, 1])
---


### 1.6 Memory-Efficient PyTorch Dataset

The complete BODMAS training dataset contains 107,548 samples, each represented by 2,381 numerical features.

To limit unnecessary RAM usage, we use a custom PyTorch Dataset that accesses the preprocessed NumPy arrays through memory mapping.

Each sample is converted into a `float32` tensor when requested. Binary labels are also converted to `float32` with shape `[1]` to match the MLP's single-output configuration.

In [37]:
class BODMASDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        x = torch.tensor(self.features[index], dtype=torch.float32)
        y = torch.tensor([self.labels[index]], dtype=torch.float32)
        return x, y

In [38]:
train_dataset = BODMASDataset(X_train, y_train)
test_dataset = BODMASDataset(X_test, y_test)

print("Training samples:", len(train_dataset))
print("Testing samples:", len(test_dataset))

sample_x, sample_y = train_dataset[0]

print("One sample features:", sample_x.shape)
print("One sample label:", sample_y.shape)
print("Feature dtype:", sample_x.dtype)
print("Label dtype:", sample_y.dtype)

Training samples: 107548
Testing samples: 26887
One sample features: torch.Size([2381])
One sample label: torch.Size([1])
Feature dtype: torch.float32
Label dtype: torch.float32


### 1.7 Training and Testing DataLoaders

We create PyTorch DataLoaders to organize the BODMAS dataset into batches.

A batch size of 32 is selected as an initial configuration to keep memory usage manageable.

The training DataLoader uses shuffling to vary the order of samples during training. The testing DataLoader does not shuffle because evaluation does not require random ordering.

In [39]:
batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

print("Training batches:", len(train_loader))
print("Testing batches:", len(test_loader))

Training batches: 3361
Testing batches: 841


In [40]:
batch_X, batch_y = next(iter(train_loader))

print("Batch features:", batch_X.shape)
print("Batch labels:", batch_y.shape)
print("Features dtype:", batch_X.dtype)
print("Labels dtype:", batch_y.dtype)

Batch features: torch.Size([32, 2381])
Batch labels: torch.Size([32, 1])
Features dtype: torch.float32
Labels dtype: torch.float32


### 1.8 Final DataLoader Validation

Before passing the prepared data to the MLP training stage, we verify that a sampled batch contains finite numerical features and valid binary labels.

This check helps identify potential input issues before model training.

In [41]:
print("All features finite:", torch.isfinite(batch_X).all().item())

print("Labels are binary:",
      torch.all((batch_y == 0) | (batch_y == 1)).item())

print("Batch feature shape:", tuple(batch_X.shape))
print("Batch label shape:", tuple(batch_y.shape))

All features finite: True
Labels are binary: True
Batch feature shape: (32, 2381)
Batch label shape: (32, 1)


### 1.9 Testing DataLoader Verification

We verify that the testing DataLoader produces batches with the expected feature dimensions, label shapes, and data types.

This ensures that both the training and testing pipelines provide compatible inputs for the MLP classifier.

In [42]:
test_batch_X, test_batch_y = next(iter(test_loader))

print("Test batch features:", test_batch_X.shape)
print("Test batch labels:", test_batch_y.shape)
print("Feature dtype:", test_batch_X.dtype)
print("Label dtype:", test_batch_y.dtype)

Test batch features: torch.Size([32, 2381])
Test batch labels: torch.Size([32, 1])
Feature dtype: torch.float32
Label dtype: torch.float32


### 1.10 Team Handoff

**Student 2 — Model Training.** Use `train_loader`. Each batch has `torch.float32` features of shape `[batch_size, 2381]` and `torch.float32` labels of shape `[batch_size, 1]`. The BODMAS labels are `0` for benign and `1` for malware. Class `1` is the positive class. [Official BODMAS label definition](https://github.com/whyisyoung/BODMAS/blob/gh-pages/index.html#L110)

**Student 3 — Evaluation.** Use `test_loader`, which has `shuffle=False` and preserves the order of the test split. The last batch may contain fewer than 32 samples. Keep each batch's predictions aligned with the labels returned in that same batch.